Cyber threats are a growing concern for organizations worldwide. These threats take many forms, including malware, phishing, and denial-of-service (DOS) attacks, compromising sensitive information and disrupting operations. The increasing sophistication and frequency of these attacks make it imperative for organizations to adopt advanced security measures. Traditional threat detection methods often fall short due to their inability to adapt to new and evolving threats. This is where deep learning models come into play.

Deep learning models can analyze vast amounts of data and identify patterns that may not be immediately obvious to human analysts. By leveraging these models, organizations can proactively detect and mitigate cyber threats, safeguarding their sensitive information and ensuring operational continuity.

As a cybersecurity analyst, you identify and mitigate these threats. In this project, you will design and implement a deep learning model to detect cyber threats. The BETH dataset simulates real-world logs, providing a rich source of information for training and testing your model. The data has already undergone preprocessing, and we have a target label, `sus_label`, indicating whether an event is malicious (1) or benign (0).

By successfully developing this model, you will contribute to enhancing cybersecurity measures and protecting organizations from potentially devastating cyber attacks.


### The Data

| Column     | Description              |
|------------|--------------------------|
|`processId`|The unique identifier for the process that generated the event - int64 |
|`threadId`|ID for the thread spawning the log - int64|
|`parentProcessId`|Label for the process spawning this log - int64|
|`userId`|ID of user spawning the log|Numerical - int64|
|`mountNamespace`|Mounting restrictions the process log works within - int64|
|`argsNum`|Number of arguments passed to the event - int64|
|`returnValue`|Value returned from the event log (usually 0) - int64|
|`sus_label`|Binary label as suspicous event (1 is suspicious, 0 is not) - int64|

More information on the dataset: [BETH dataset](accreditation.md)

In [69]:
# Import required libraries
import pandas as pd
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.functional as functional
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
from torchmetrics import Accuracy
# from sklearn.metrics import accuracy_score  # uncomment to use sklearn

In [70]:
# Load preprocessed data
train_df = pd.read_csv('labelled_train.csv')
test_df = pd.read_csv('labelled_test.csv')
val_df = pd.read_csv('labelled_validation.csv')

# View the first 5 rows of training set
train_df.head()

,processId,threadId,parentProcessId,userId,mountNamespace,argsNum,returnValue,sus_label
0,381,7337,1,100,4026532231,5,0,1
1,381,7337,1,100,4026532231,1,0,1
2,381,7337,1,100,4026532231,0,0,1
3,7347,7347,7341,0,4026531840,2,-2,1
4,7347,7347,7341,0,4026531840,4,0,1


In [71]:
#Get Xs and Ys
x_train = train_df.iloc[:, :-1].to_numpy()
y_train = train_df.iloc[:, -1].to_numpy()
x_test = test_df.iloc[:, :-1].to_numpy()
y_test = test_df.iloc[:, -1].to_numpy()
x_val = val_df.iloc[:, :-1].to_numpy()
y_val = val_df.iloc[:, -1].to_numpy()

In [72]:
#Scale Data
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)
x_val = scaler.transform(x_val)

In [73]:
#Convert Data to Tensors
x_train_tensor = torch.tensor(x_train).float()
y_train_tensor = torch.tensor(y_train).float().view(-1, 1)
x_test_tensor = torch.tensor(x_test).float()
y_test_tensor = torch.tensor(y_test).float().view(-1, 1)
x_val_tensor = torch.tensor(x_val).float()
y_val_tensor = torch.tensor(y_val).float().view(-1, 1)

In [74]:
#Create DNN Model Structure
class Net(nn.Module):
    def __init__(self, input_feature_count: int):
        super().__init__()
        self.fc1 = nn.Linear(input_feature_count, 100)
        self.fc2 = nn.Linear(100, 50)
        self.fc3 = nn.Linear(50, 1)

    def forward(self, x):
        x = nn.functional.elu(self.fc1(x))
        x = nn.functional.elu(self.fc2(x))
        x = nn.functional.sigmoid(self.fc3(x))
        return x

model = Net(x_train_tensor.shape[1])
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)
acc = Accuracy(task = "binary")

In [75]:
#Training
epoch_count = 10

for epoch in range(epoch_count):
    model.train()
    optimizer.zero_grad()
    train_preds = model(x_train_tensor)
    loss = criterion(train_preds, y_train_tensor)
    loss.backward()
    optimizer.step()

In [76]:
#Validation
model.eval()
with torch.no_grad():
    train_preds = model(x_train_tensor)
    val_preds = model(x_val_tensor)
    test_preds = model(x_test_tensor)
    acc_train = acc(train_preds, y_train_tensor)
    acc_val = acc(val_preds, y_val_tensor)
    acc_test = acc(test_preds, y_test_tensor)

In [77]:
train_accuracy = acc_train.item()
val_accuracy = acc_val.item()
test_accuracy = acc_test.item()

print ("Train Accuracy: ", train_accuracy)
print ("Val Accuracy: ", val_accuracy)
print ("Test Accuracy: ", test_accuracy)

Train Accuracy:  0.9983371496200562
Val Accuracy:  0.9958405494689941
Test Accuracy:  0.09265109896659851
